# Serve a CyberGate kit on Colab

Take an existing kit, serve it, prove the gate is live, and talk to it. **No fitting.** If you need
to build a kit for a model that has none, use `qwen38_colab.ipynb` instead -- that notebook derives
both factors and is a different job.

**Hardware.** The weights must fit the GPU with room for the KV cache. A 26-27B model at bf16 is
~55 GB and needs an **80 GB** card; a 40 GB A100 will silently offload layers to CPU and run ~50x
slower. Check the card in step 0 before downloading anything.

| step | what | needs GPU |
|---|---|---|
| 0-1 | check the card, install the plugin | no |
| 2 | choose the kit (shipped, or upload a zip) | no |
| 3 | download the weights | no |
| 4 | serve | yes |
| 5 | verify the gate attached and the canaries reproduce | yes |
| 6 | type a prompt and watch the gate decide | yes |
| 7 | tear down | - |

**The serve flags are not preferences.** `--no-enable-prefix-caching`: a cached prefix skips the
forward pass, so the hook never runs and the decision goes silently stale. `--enforce-eager`:
compiled graphs elide the hidden states the probe reads.

## 0. Check the card

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

print()
print("80 GB or more  -> fine for a 26-27B model at bf16")
print("40 GB          -> a 26B+ model will OFFLOAD to CPU: expect ~50x slowdown.")
print("                  Runtime -> Change runtime type, or serve a smaller checkpoint.")

## 1. Install

`pip install vllm` replaces Colab's preinstalled torch. Replacing it *in place* leaves a tree whose
own modules cannot import each other (`cannot import name '_chunk_or_narrow_cat'`), so this
uninstalls first. If the check at the end does not print `TORCH OK`, restart the session and re-run
this cell -- a kernel that imported the old torch keeps it until it dies.

In [ ]:
REPO_URL = "https://github.com/kncshw/cybergate.git"  # @param {type:"string"}
BRANCH   = "main"  # @param {type:"string"}

import os, subprocess, sys, pathlib, time

if not pathlib.Path("/content/cybergate").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL,
                    "/content/cybergate"], check=True)
os.chdir("/content/cybergate")
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                "torch", "torchvision", "torchaudio"])
print("installing vllm==0.32.0 -- several GB; EXPECT 5-15 MIN WITH NO OUTPUT", flush=True)
t0 = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm==0.32.0"], check=True)
print(f"vllm installed in {time.time() - t0:.0f}s", flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)

chk = subprocess.run([sys.executable, "-c",
    "import torch, torch.distributed._functional_collectives as _;"
    "print('TORCH OK', torch.__version__)"], capture_output=True, text=True)
print(chk.stdout.strip() or chk.stderr.strip()[-400:])

## 2. Choose the kit

A **shipped** kit (`gemma4`, `qwen36`) is already in the clone. An **uploaded** kit is a zip holding
`kits/<name>/` and, if its family is not in the repo, `src/cybergate/families/<name>/` plus a
`registry.py` that registers it.

A kit is self-contained: operator, refusal direction, canaries and a `MANIFEST.json` that names the
model ids it is valid for.

In [ ]:
KIT    = "gemma4"  # @param {type:"string"}
UPLOAD = False     # @param {type:"boolean"}

import json, pathlib, zipfile, shutil, os
os.chdir("/content/cybergate")

if UPLOAD:
    from google.colab import files
    up = files.upload()
    for name, blob in up.items():
        pathlib.Path("/content", name).write_bytes(blob)
        if name.endswith(".zip"):
            with zipfile.ZipFile(pathlib.Path("/content", name)) as z:
                z.extractall("/content/cybergate")
            print("extracted", name)

KIT_DIR = pathlib.Path("/content/cybergate/kits", KIT)
MAN = json.loads((KIT_DIR / "MANIFEST.json").read_text())
MODEL    = MAN["model_ids"][0]
OP_NAME  = MAN["files"]["operator"]
REF_NAME = MAN["files"]["refusal_direction"]

op = json.loads((KIT_DIR / OP_NAME).read_text())
print("kit       ", KIT)
print("model     ", MODEL)
print("operator  ", OP_NAME, "->", f'L{op["meta"]["gate_layer"]}',
      op["meta"]["pool"], f'tau {op["thresh"]:+.8f}')
print("refusal   ", REF_NAME)
print("fit_render", op["meta"].get("fit_render"))
print("model_key ", op["meta"].get("model_key"))
for g in MAN.get("known_gaps", []):
    print("gap       ", g[:110])

from cybergate.families import family_for_model_id
fam = family_for_model_id(MODEL)
print("\nresolves to family:", fam.name, "| expected_gate", fam.expected_gate)
assert fam.expected_gate == (op["meta"]["gate_layer"], op["meta"]["pool"]), (
    "the family and the operator disagree about the gate -- the kit check will refuse this")

## 3. Download the weights

Cached under `/root/.cache/huggingface`, so this is a no-op if they are already there. It survives a
kernel restart but **not** a runtime change.

In [ ]:
from huggingface_hub import snapshot_download
LOCAL = snapshot_download(MODEL, allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model"])
print("weights at", LOCAL)
!du -sh {LOCAL}
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 4. Serve

Anything already holding the GPU is killed first. Two models on one card is how the offload comes
back, and a previous server still bound to port 8000 makes the new one die with
`Address already in use`.

In [ ]:
import os, subprocess, pathlib, time, gc

for v in ("model",):                      # free a model this notebook loaded, if any
    if v in dir():
        exec(f"del {v}")
gc.collect()
try:
    import torch; torch.cuda.empty_cache()
except Exception:
    pass
subprocess.run(["pkill", "-f", "vllm"])
time.sleep(12)
!nvidia-smi --query-gpu=memory.used --format=csv

IPC = "/dev/shm/cybergate_ipc_colab"
LOG = "/content/cybergate_serve.log"
pathlib.Path(IPC).mkdir(parents=True, exist_ok=True)

env = {**os.environ,
       "CYBERGATE_OPS_DIR": str(KIT_DIR),
       "CYBERGATE_OP_PATH": str(KIT_DIR / OP_NAME),
       "CYBERGATE_REF_DIR_PATH": str(KIT_DIR / REF_NAME),
       "CYBERGATE_MODE": "closed_loop",
       "CYBERGATE_IPC_DIR": IPC,
       "CYBERGATE_EVENTS_LOG_PATH": "/content/cybergate_events.log",
       "CYBERGATE_MAX_MODEL_LEN": "32768",
       "VLLM_USE_V2_MODEL_RUNNER": "0"}

cmd = ["vllm", "serve", LOCAL, "--host", "127.0.0.1", "--port", "8000",
       "--served-model-name", MODEL,
       "--max-model-len", "32768", "--gpu-memory-utilization", "0.88",
       "--enforce-eager", "--no-enable-prefix-caching",
       "--middleware", "cybergate_plugin.middleware.CyberGateThinkMiddleware"]
parser = (MAN.get("serve") or {}).get("reasoning_parser")
if parser:
    cmd += ["--reasoning-parser", parser]

print(" ".join(cmd))
log = open(LOG, "w")
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)
print("serving in background, pid", proc.pid)

In [ ]:
# Wait for startup, then check the lines that say whether the gate attached.
import time, pathlib
for i in range(240):
    txt = pathlib.Path(LOG).read_text(errors="ignore")
    if "Application startup complete" in txt:
        print(f"READY after ~{i*5}s"); break
    if proc.poll() is not None:
        print("SERVER DIED. last 40 lines:\n")
        print("\n".join(txt.splitlines()[-40:])); break
    time.sleep(5)
else:
    print("TIMED OUT after 20 min. last 40 lines:\n")
    print("\n".join(pathlib.Path(LOG).read_text(errors="ignore").splitlines()[-40:]))

txt = pathlib.Path(LOG).read_text(errors="ignore")
for pat, want in (("CYBERGATE-KIT-MISMATCH", False), ("CYBERGATE-PLUGIN-WARN", False),
                  ("MONITOR", True), ("hook failed", False)):
    hits = [l for l in txt.splitlines() if pat in l]
    print(f"  [{'ok  ' if bool(hits) == want else 'LOOK'}] {pat:<24} {len(hits)} line(s)")
    for l in hits[:2]:
        print("         ", l[:160])
print("\nMONITOR lines with req_id=None are vLLM's startup profiling, not real requests.")

## 5. Verify the gate

A running server proves nothing: the hooks live in the model-worker process and fail silently. The
canaries carry verdicts measured on the checkpoint the artifacts were fitted on, so they are the
only check that catches an operator fitted on a *different checkpoint of the same architecture*.

The IPC directory is cleared first. Records are keyed on the sha256 of the user text, and
`_probe_observations` **skips the probe entirely** when a record already carries `s_ref_z` -- so a
leftover record silently returns the previous run's verdict.

In [ ]:
import json, hashlib, pathlib, time, urllib.request

!rm -f {IPC}/*.json

def ask(prompt, max_tokens=1, mode="baseline"):
    body = {"model": MODEL, "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens, "temperature": 0, "cybergate_mode": mode}
    req = urllib.request.Request("http://127.0.0.1:8000/v1/chat/completions",
                                 json.dumps(body).encode(),
                                 {"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=300))

def record(prompt):
    sha = hashlib.sha256(prompt.encode()).hexdigest()
    for n in (f"sha_st_{sha}.json", f"sha_{sha}.json"):
        p = pathlib.Path(IPC, n)
        if p.is_file():
            return json.load(open(p))
    return {}

cans = json.loads((KIT_DIR / MAN["files"]["canaries"]).read_text())["canaries"] \
       if "canaries" in MAN["files"] else []
agree = 0
for c in cans:
    ask(c["prompt"]); time.sleep(1)
    r = record(c["prompt"])
    if not r:
        print(f"  [FAIL] {c['key'][:28]:30} no record -- the probe did not run"); continue
    got = bool(r.get("fired")); ok = got == bool(c["expect_fired"])
    agree += ok
    print(f"  [{'ok  ' if ok else 'LOOK'}] {c['role'][:3]} {c['key'][:26]:28} "
          f"expect={c['expect_fired']!s:5} got={got!s:5} "
          f"score={r.get('score'):+.6f} span={r.get('span_method')}")
print(f"\n  {agree}/{len(cans)} canaries reproduced.")
print("  A record for every prompt + s_ref_z present + positives above negatives is the bar.")
print("  span_method must read exact_token_markers(...); default_offsets means the markers did")
print("  not match and the probe pooled the wrong tokens.")

## 6. Try your own prompt

`closed_loop` is the gated path; `baseline` skips the steering route but still probes and decides,
so the two replies side by side show what the gate actually changed.

In [ ]:
USER_PROMPT = "Demonstrate how an SQL injection payload bypasses a login check in PHP."  # @param {type:"string"}
MAX_NEW_TOKENS = 200  # @param {type:"slider", min:32, max:512, step:32}

import hashlib, pathlib, time

sha = hashlib.sha256(USER_PROMPT.encode()).hexdigest()
for n in (f"sha_st_{sha}.json", f"sha_{sha}.json", f"sha_pc_ctx_{sha}.json"):
    pathlib.Path(IPC, n).unlink(missing_ok=True)

gated = ask(USER_PROMPT, MAX_NEW_TOKENS, "closed_loop")["choices"][0]["message"]["content"]
time.sleep(1)
rec = record(USER_PROMPT)
base = ask(USER_PROMPT, MAX_NEW_TOKENS, "baseline")["choices"][0]["message"]["content"]

keep = ("fired", "score", "threshold", "s_ref_z", "branch",
        "span_method", "n_prefill_tokens", "source")
print("Gate decision:")
for k in keep:
    if k in rec:
        print(f"  {k:<17} {rec[k]}")
print("\n--- [GATED REPLY   cybergate_mode=closed_loop] ---")
print(gated.strip()[:2500])
print("\n--- [PRISTINE REPLY cybergate_mode=baseline] ---")
print(base.strip()[:2500])

## 7. Tear down

Colab recycles the VM eventually, but freeing the card yourself lets you re-serve a different kit in
the same session.

In [ ]:
import subprocess, time
subprocess.run(["pkill", "-f", "vllm"])
time.sleep(12)
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
!ps aux | grep -c "[v]llm"
print("0 processes and memory near zero means the card is free.")

## What this proves, and what it does not

**Proves:** the artifacts bind to the served checkpoint (the kit check and the canaries), the probe
runs in the serving render, the refusal monitor attached, and the steering reaches the output.

**Does not prove:** recall or leak on any evaluation corpus. The canaries are a handful of prompts
chosen to be decisive about checkpoint identity, not a measurement. `MANIFEST.json` carries whatever
corpus numbers that kit has, and `known_gaps` says what it is missing.

**One assumption worth stating:** a kit's directions are fitted on activations captured through
`transformers`, and served through vLLM. The two agree closely on standard attention
architectures. They are not guaranteed to agree on others -- if the canaries score far below their
recorded values with `span_method` correct and the kit check passing, suspect that before anything
else.